In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
JUDGE_LOG = '../data/judged_log.csv'

df = pd.read_csv(JUDGE_LOG)
print(f'\nTotal rows loaded: {len(df)}')
df.head(3)

## 1. General Overview

In [ ]:
total       = len(df)
approved    = (df['status'] == 'APPROVED').sum()
rejected    = (df['status'] == 'REJECTED').sum()
rewritten   = df['rewritten'].sum() if df['rewritten'].dtype == bool else (df['rewritten'] == True).sum()
print(f'  Totale eventi valutati : {total}')
print(f'  APPROVED               : {approved:>6}  ({approved/total*100:.1f}%)')
print(f'  REJECTED               : {rejected:>6}  ({rejected/total*100:.1f}%)')
print(f'  Riscritti (rewritten)  : {rewritten:>6}  ({rewritten/total*100:.1f}%)')


## 2. Score Distribution

In [ ]:
print('Score avg (all judges)')
print(df['score_avg_all'].describe().round(2))
print('\nScore avg (only approved)')
print(df['score_avg_approvers'].describe().round(2))
print('\nScore avg per status')
print(df.groupby('status')[['score_avg_all', 'score_avg_approvers']].mean().round(2))

## 3. Approvals Count Distribution

In [ ]:
counts = df['approvals_count'].value_counts().sort_index()
print('approvals_count | n events | %')
for val, cnt in counts.items():
    label = {0: 'all reject', 1: 'minority', 2: 'majority', 3: 'unanimity'}.get(val, str(val))
    print(f'  {val} ({label:<13}) | {cnt:>6}   | {cnt/total*100:.1f}%')

## 4. Judge Agreement Rate

In [ ]:
unanimous_approve = (df['approvals_count'] == 3).sum()
unanimous_reject  = (df['approvals_count'] == 0).sum()
split             = ((df['approvals_count'] == 1) | (df['approvals_count'] == 2)).sum()
print(f'Unanimity approve  : {unanimous_approve:>5}  ({unanimous_approve/total*100:.1f}%)')
print(f'Unanimity reject   : {unanimous_reject:>5}  ({unanimous_reject/total*100:.1f}%)')
print(f'Split (1 o 2/3)    : {split:>5}  ({split/total*100:.1f}%)')
# Agreement rate = how many times the 3 judges agree
agreement_rate = (unanimous_approve + unanimous_reject) / total * 100
print(f'\nAgreement total rate: {agreement_rate:.1f}%')

## 5. Individual Judge Performance

In [ ]:
judge_stats = []
for j in ['judge1', 'judge2', 'judge3']:
    provider_col = f'{j}_provider'
    vote_col     = f'{j}_vote'
    score_col    = f'{j}_score'
    if provider_col not in df.columns:
        continue
    for provider, group in df.groupby(provider_col):
        if not provider:
            continue
        n          = len(group)
        approves   = (group[vote_col] == 'approve').sum()
        rejects    = (group[vote_col] == 'reject').sum()
        errors     = (group[vote_col] == 'error').sum()
        avg_score  = group[score_col].mean()
        judge_stats.append({
            'slot':       j,
            'provider':   provider,
            'n':          n,
            'approve_%':  round(approves / n * 100, 1),
            'reject_%':   round(rejects  / n * 100, 1),
            'error_%':    round(errors   / n * 100, 1),
            'avg_score':  round(avg_score, 2),
        })
stats_df = pd.DataFrame(judge_stats).drop_duplicates(subset='provider')
print(stats_df.to_string(index=False))

## 6. Top Rejection Reasons by Provider

In [ ]:
from collections import Counter
def extract_criterion(reason):
    if not isinstance(reason, str):
        return 'UNKNOWN'
    reason = reason.strip()
    for criterion in ['INTEGRITY', 'DERIVABILITY', 'CONSISTENCY', 'PROPORTIONALITY']:
        if criterion in reason.upper():
            return criterion
    if 'all criteria met' in reason.lower():
        return 'ALL_MET'
    return 'OTHER'
for j in ['judge1', 'judge2', 'judge3']:
    prov_col = f'{j}_provider'
    vote_col = f'{j}_vote'
    reas_col = f'{j}_reason'
    if prov_col not in df.columns:
        continue
    for provider, group in df.groupby(prov_col):
        if not provider:
            continue
        rejects = group[group[vote_col] == 'reject'][reas_col]
        if rejects.empty:
            continue
        criteria = Counter(rejects.apply(extract_criterion))
        print(f'\n{provider.upper()} reject reasons')
        for crit, cnt in criteria.most_common():
            print(f'  {crit:<20} : {cnt:>4}  ({cnt/len(rejects)*100:.1f}%)')

## 7. Approval Policy Simulation

In [ ]:
policies = [
    ('Majority vote (>=2)  + avg_all>=60',   lambda r: r['approvals_count'] >= 2 and r['score_avg_all'] >= 60),
    ('Majority vote (>=2)  only',            lambda r: r['approvals_count'] >= 2),
    ('Majority vote (>=2)  + avg_appr>=70',  lambda r: r['approvals_count'] >= 2 and r['score_avg_approvers'] >= 70),
    ('Unanimity (3/3)',                      lambda r: r['approvals_count'] == 3),
    ('At least 1 approve',                     lambda r: r['approvals_count'] >= 1),
]
print(f'{"Policy":<45} | {"Approved":>8} | {"% total":>9}')
for name, fn in policies:
    n = df.apply(fn, axis=1).sum()
    print(f'{name:<45} | {n:>8} | {n/total*100:>8.1f}%')

## 8. Top 20 Most Contested Approved Events

In [ ]:
contested = df[df['status'] == 'APPROVED'].nsmallest(20, 'score_avg_all')
print(contested[['original_row_id', 'event', 'approvals_count',
                  'score_avg_all', 'score_avg_approvers']].to_string(index=False))

## 9. Summary Export

In [ ]:
summary = {
    'total_events':        total,
    'approved':            int(approved),
    'rejected':            int(rejected),
    'rewritten':           int(rewritten),
    'approval_rate_%':     round(approved / total * 100, 2),
    'agreement_rate_%':    round(agreement_rate, 2),
    'avg_score_all':       round(df['score_avg_all'].mean(), 2),
    'avg_score_approvers': round(df['score_avg_approvers'].mean(), 2),
    'unanimous_approve':   int(unanimous_approve),
    'unanimous_reject':    int(unanimous_reject),
    'split_vote':          int(split),
}
summary_df = pd.DataFrame([summary]).T.rename(columns={0: 'value'})
summary_df.to_csv('../data/judgement_summary.csv')
print('Saved in data/judgement_summary.csv')
print(summary_df)